In [1]:
!bedtools intersect -a "UCE_hits_coordinates.bed" -b UCE_refseq_genes.bed -wa -wb > 0.05_D21_mapped_regions.bed

In [2]:
import pandas as pd
def process_mapped_regions(input_csv, output_csv):
    # Read the input file
    df = pd.read_csv(input_csv, header=None)

    # Define column names for the 14-column file
    df.columns = [
        "query_chrom", "query_start", "query_end",   # Query region
        "transcript_chrom", "transcript_start", "transcript_end", "transcript_id",  # Transcript details
        "score", "strand", "cds_start", "cds_end",  # CDS details
        "rgb", "exon_count", "exon_lengths", "exon_starts"  # Exon information
    ]

    # Prepare the output data
    output_rows = []

    for _, row in df.iterrows():
        # Parse the exon start offsets and lengths
        relative_exon_starts = list(map(int, row["exon_starts"].strip(",").split(",")))
        exon_lengths = list(map(int, row["exon_lengths"].strip(",").split(",")))

        # Calculate absolute exon start and end positions
        transcript_start = row["transcript_start"]
        exon_starts = [transcript_start + offset for offset in relative_exon_starts]
        exon_ends = [start + length for start, length in zip(exon_starts, exon_lengths)]

        # Query region
        query_start = row["query_start"]
        query_end = row["query_end"]

        # Check if the query overlaps any exon
        region_type = "Intron"
        for exon_start, exon_end in zip(exon_starts, exon_ends):
            if exon_start <= query_start <= exon_end or exon_start <= query_end <= exon_end:
                region_type = "Exon"
                break

        # Append results
        output_rows.append({
            "Query Coordinates": f"{row['query_chrom']}:{query_start}-{query_end}",
            "Transcript ID": row["transcript_id"],
            "Region Type": region_type
        })

    # Convert to DataFrame and save
    output_df = pd.DataFrame(output_rows)
    output_df.to_csv(output_csv, index=False)
    print(f"Output saved to {output_csv}")


# Example usage
input_csv = "UCEhits_mapped_regions.csv"  # Replace with your input file
output_csv = "UCEhits_region_types.csv"   # Replace with your desired output file
process_mapped_regions(input_csv, output_csv)



Output saved to UCEhits_region_types.csv


In [3]:
!grep -F -f canonical_transcript_ids.txt UCEhits_region_types.csv > UCEhits_regions_type_canonical.csv


In [4]:
import pandas as pd
import requests
import time

def get_gene_info(transcript_id):
    """
    Fetch gene information from Ensembl REST API using the transcript ID.
    If the gene description is not found directly, fetch it using the gene ID.
    """
    server = "https://rest.ensembl.org"
    transcript_endpoint = f"/lookup/id/{transcript_id}?expand=1"
    headers = {"Content-Type": "application/json"}
    
    try:
        # Query transcript information
        response = requests.get(server + transcript_endpoint, headers=headers, timeout=10)
        if response.ok:
            transcript_data = response.json()
            gene_name = transcript_data.get("display_name", "N/A")
            gene_id = transcript_data.get("Parent", None)  # Get Parent gene ID
            
            # If gene ID is available, fetch gene description
            if gene_id:
                gene_description = get_gene_description(gene_id)
            else:
                gene_description = "N/A"
            
            return gene_name, gene_description
        else:
            print(f"Error: {response.status_code} for Transcript ID: {transcript_id}")
            return "N/A", "N/A"
    except requests.exceptions.RequestException as e:
        print(f"Request failed for Transcript ID: {transcript_id} - {e}")
        return "N/A", "N/A"

def get_gene_description(gene_id):
    """
    Fetch gene description using the gene ID.
    """
    server = "https://rest.ensembl.org"
    gene_endpoint = f"/lookup/id/{gene_id}?expand=1"
    headers = {"Content-Type": "application/json"}
    
    try:
        # Query gene information
        response = requests.get(server + gene_endpoint, headers=headers, timeout=10)
        if response.ok:
            gene_data = response.json()
            return gene_data.get("description", "N/A")
        else:
            print(f"Error: {response.status_code} for Gene ID: {gene_id}")
            return "N/A"
    except requests.exceptions.RequestException as e:
        print(f"Request failed for Gene ID: {gene_id} - {e}")
        return "N/A"

def enrich_gene_information(input_csv, output_csv):
    """
    Enrich the region_type.csv file with gene name and annotation from Ensembl.
    """
    # Read the input CSV
    df = pd.read_csv(input_csv,header=None)
    # Define column names for the 14-column file
    df.columns = [
        "Query Coordinates","Transcript ID","Region Type",
    ]
    
    df['Transcript ID'] = df['Transcript ID'].str.split('.').str[0]
    
    
    # Create columns for gene name and annotation
    df["Gene Name"] = ""
    df["Gene Annotation"] = ""
    
    # Loop through transcript IDs and fetch information
    for index, row in df.iterrows():
        transcript_id = row["Transcript ID"]
        print(f"Processing Transcript ID: {transcript_id}...")
        gene_name, gene_annotation = get_gene_info(transcript_id)
        df.at[index, "Gene Name"] = gene_name
        df.at[index, "Gene Annotation"] = gene_annotation
        # Pause to respect API limits
        time.sleep(0.2)
    
    # Save the enriched file
    df.to_csv(output_csv, index=False)
    print(f"Enriched data saved to {output_csv}")

In [5]:
# Example usage
input_csv = "UCEhits_regions_type_canonical.csv"  # Update with the path to your input file
output_csv = "UCEhits_regions_type_final.csv"  # Desired output file
enrich_gene_information(input_csv, output_csv)

Processing Transcript ID: ENST00000221855...
Enriched data saved to UCEhits_regions_type_final.csv


In [7]:
import pandas as pd

def two_files_mapping_combined(df_1_path, df_2_path):
    # Load both DataFrames
    df_1 = pd.read_csv(df_1_path)
    df_2 = pd.read_csv(df_2_path)
    df_1['gRNA-ID_start'] = df_1['gRNA-ID_start'].fillna(0).astype(int)
    df_1['gRNA-ID_end'] = df_1['gRNA-ID_end'].fillna(0).astype(int)

    # Create "temp_Query Coordinates" in df_1 to match df_2's format
    df_1['temp_Query Coordinates'] = (
        df_1['chr'].astype(str) + ':' + 
        df_1['gRNA-ID_start'].astype(str) + '-' + 
        df_1['gRNA-ID_end'].astype(str)
    )

    # Merge the DataFrames on the new temporary column
    merged_df = pd.merge(
        df_1,
        df_2,
        left_on='temp_Query Coordinates',
        right_on='Query Coordinates',
        how='left'  # Keep all rows from df_1
    )

    # Drop redundant columns (optional)
    merged_df = merged_df.drop(columns=['temp_Query Coordinates', 'Query Coordinates', ])

    return merged_df

# Example usage:
merged_data = two_files_mapping_combined("A2058_D14_0.1_region_hits_details.csv", "A2058_D14_0.1_region_final.csv")
merged_data.to_csv("A2058_0.1_D14_regions_type_final2.csv", index=False)
    